# 1. Training
Hyperparameter search, the proposed model on the five splits, and the same network with other pooling functions.

Run the cells from top to bottom. A run whose result file already exists is skipped, so the notebook can be stopped and started again.

In [1]:
import torch

from constants import *
from utils import get_device
from experiments import run_experiment
from hparam_search import run_hyperparameter_search
from disk_utils import load_results
from result_tables import per_seed_table, summary_table

In [2]:
USE_GPU = True
device = get_device(USE_GPU)
print(device)

# QUICK_TEST = True makes a tiny dry run of every step (1 epoch, 3 batches)
# and writes to results_quick_test/. Use it to check the setup.
QUICK_TEST = False

if QUICK_TEST:
    results_dir = QUICK_TEST_RESULTS_DIR
    num_epochs = 1
    max_train_batches = 3
else:
    results_dir = RESULTS_DIR
    num_epochs = NUM_EPOCHS
    max_train_batches = None

if QUICK_TEST:
    num_search_trials = 2
else:
    num_search_trials = NUM_SEARCH_TRIALS

cuda


## Hyperparameter search
Random search on split 0. Trial 0 is the configuration that was fixed beforehand. The configurations are compared by validation loss; the test subset is not used.

In [3]:
search_table = run_hyperparameter_search(
    device,
    num_trials=num_search_trials,
    seed=0,
    num_epochs=num_epochs,
    max_train_batches=max_train_batches,
    results_dir=results_dir
)

[hp_00 s0] Epoch 01 | Train Loss: 1.8495, Train Acc: 0.3616 | Val Loss: 1.0340, Val Acc: 0.6609 | LR: 1.0e-03
[hp_00 s0] Epoch 02 | Train Loss: 0.7107, Train Acc: 0.7454 | Val Loss: 0.4616, Val Acc: 0.8506 | LR: 1.0e-03
[hp_00 s0] Epoch 03 | Train Loss: 0.4200, Train Acc: 0.8573 | Val Loss: 0.3807, Val Acc: 0.8333 | LR: 1.0e-03
[hp_00 s0] Epoch 04 | Train Loss: 0.3366, Train Acc: 0.8708 | Val Loss: 0.3185, Val Acc: 0.8966 | LR: 1.0e-03
[hp_00 s0] Epoch 05 | Train Loss: 0.2938, Train Acc: 0.9065 | Val Loss: 0.2570, Val Acc: 0.8851 | LR: 1.0e-03
[hp_00 s0] Epoch 06 | Train Loss: 0.2657, Train Acc: 0.9139 | Val Loss: 0.1978, Val Acc: 0.9253 | LR: 1.0e-03
[hp_00 s0] Epoch 07 | Train Loss: 0.2307, Train Acc: 0.9151 | Val Loss: 0.2035, Val Acc: 0.9368 | LR: 1.0e-03
[hp_00 s0] Epoch 08 | Train Loss: 0.1711, Train Acc: 0.9520 | Val Loss: 0.2034, Val Acc: 0.9310 | LR: 1.0e-03
[hp_00 s0] Epoch 09 | Train Loss: 0.1543, Train Acc: 0.9520 | Val Loss: 0.2866, Val Acc: 0.9023 | LR: 1.0e-03
[hp_00 s0]

In [4]:
search_table

,lstm_hidden,emb_dim,learning_rate,dropout,batch_size,trial,best_epoch,val_loss,val_acc,n_params,seconds
16,64,32,0.0010,0.3,8,16,12,0.154977,0.948276,126892,111
19,128,32,0.0010,0.2,8,19,14,0.172298,0.948276,333676,121
18,256,32,0.0010,0.2,16,18,14,0.172786,0.931034,993004,68
4,256,32,0.0010,0.2,8,4,6,0.179699,0.936782,993004,73
6,128,64,0.0005,0.2,8,6,12,0.180111,0.936782,499532,111
7,64,64,0.0010,0.2,8,7,8,0.180492,0.948276,210828,89
1,64,32,0.0010,0.3,16,1,11,0.180583,0.931034,126892,62
0,128,32,0.0010,0.3,8,0,12,0.183518,0.936782,333676,110
8,128,32,0.0005,0.3,16,8,16,0.184491,0.936782,333676,77
11,256,64,0.0005,0.5,16,11,9,0.194752,0.925287,1322700,52


## Proposed model
BiLSTM with attention on all five note attributes (pitch class, accidental, octave, duration, measure position).

In [5]:
for seed in SEEDS:
    run_experiment(
        tag="full",
        seed=seed,
        device=device,
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[full s0] Epoch 01 | Train Loss: 1.8495, Train Acc: 0.3616 | Val Loss: 1.0340, Val Acc: 0.6609 | LR: 1.0e-03
[full s0] Epoch 02 | Train Loss: 0.7107, Train Acc: 0.7454 | Val Loss: 0.4616, Val Acc: 0.8506 | LR: 1.0e-03
[full s0] Epoch 03 | Train Loss: 0.4200, Train Acc: 0.8573 | Val Loss: 0.3807, Val Acc: 0.8333 | LR: 1.0e-03
[full s0] Epoch 04 | Train Loss: 0.3366, Train Acc: 0.8708 | Val Loss: 0.3185, Val Acc: 0.8966 | LR: 1.0e-03
[full s0] Epoch 05 | Train Loss: 0.2938, Train Acc: 0.9065 | Val Loss: 0.2570, Val Acc: 0.8851 | LR: 1.0e-03
[full s0] Epoch 06 | Train Loss: 0.2657, Train Acc: 0.9139 | Val Loss: 0.1978, Val Acc: 0.9253 | LR: 1.0e-03
[full s0] Epoch 07 | Train Loss: 0.2307, Train Acc: 0.9151 | Val Loss: 0.2035, Val Acc: 0.9368 | LR: 1.0e-03
[full s0] Epoch 08 | Train Loss: 0.1711, Train Acc: 0.9520 | Val Loss: 0.2034, Val Acc: 0.9310 | LR: 1.0e-03
[full s0] Epoch 09 | Train Loss: 0.1543, Train Acc: 0.9520 | Val Loss: 0.2866, Val Acc: 0.9023 | LR: 1.0e-03
[full s0] Epoch 10 

## Without packing
The LSTM also runs over the padding, as in the preliminary version.

In [6]:
for seed in SEEDS:
    run_experiment(
        tag="full_unpacked",
        seed=seed,
        device=device,
        packed=False,
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[full_unpacked s0] Epoch 01 | Train Loss: 1.8568, Train Acc: 0.3690 | Val Loss: 0.9784, Val Acc: 0.6897 | LR: 1.0e-03
[full_unpacked s0] Epoch 02 | Train Loss: 0.6836, Train Acc: 0.7761 | Val Loss: 0.5166, Val Acc: 0.8333 | LR: 1.0e-03
[full_unpacked s0] Epoch 03 | Train Loss: 0.4409, Train Acc: 0.8475 | Val Loss: 0.4315, Val Acc: 0.8736 | LR: 1.0e-03
[full_unpacked s0] Epoch 04 | Train Loss: 0.3328, Train Acc: 0.8672 | Val Loss: 0.3281, Val Acc: 0.8736 | LR: 1.0e-03
[full_unpacked s0] Epoch 05 | Train Loss: 0.3169, Train Acc: 0.8979 | Val Loss: 0.2303, Val Acc: 0.9253 | LR: 1.0e-03
[full_unpacked s0] Epoch 06 | Train Loss: 0.2403, Train Acc: 0.9102 | Val Loss: 0.2392, Val Acc: 0.9138 | LR: 1.0e-03
[full_unpacked s0] Epoch 07 | Train Loss: 0.2421, Train Acc: 0.9250 | Val Loss: 0.2226, Val Acc: 0.9253 | LR: 1.0e-03
[full_unpacked s0] Epoch 08 | Train Loss: 0.1883, Train Acc: 0.9336 | Val Loss: 0.2081, Val Acc: 0.9138 | LR: 1.0e-03
[full_unpacked s0] Epoch 09 | Train Loss: 0.2109, Train 

## Other pooling functions in place of attention

In [7]:
for seed in SEEDS:
    run_experiment(
        tag="mean_pool",
        seed=seed,
        device=device,
        pooling="mean",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[mean_pool s0] Epoch 01 | Train Loss: 1.8564, Train Acc: 0.3456 | Val Loss: 1.0726, Val Acc: 0.5690 | LR: 1.0e-03
[mean_pool s0] Epoch 02 | Train Loss: 0.7900, Train Acc: 0.7232 | Val Loss: 0.4827, Val Acc: 0.8391 | LR: 1.0e-03
[mean_pool s0] Epoch 03 | Train Loss: 0.4713, Train Acc: 0.8426 | Val Loss: 0.6618, Val Acc: 0.7414 | LR: 1.0e-03
[mean_pool s0] Epoch 04 | Train Loss: 0.4247, Train Acc: 0.8487 | Val Loss: 0.3645, Val Acc: 0.8678 | LR: 1.0e-03
[mean_pool s0] Epoch 05 | Train Loss: 0.3041, Train Acc: 0.9028 | Val Loss: 0.2760, Val Acc: 0.9080 | LR: 1.0e-03
[mean_pool s0] Epoch 06 | Train Loss: 0.2685, Train Acc: 0.9151 | Val Loss: 0.2899, Val Acc: 0.8908 | LR: 1.0e-03
[mean_pool s0] Epoch 07 | Train Loss: 0.2581, Train Acc: 0.9114 | Val Loss: 0.2209, Val Acc: 0.9310 | LR: 1.0e-03
[mean_pool s0] Epoch 08 | Train Loss: 0.2317, Train Acc: 0.9225 | Val Loss: 0.1885, Val Acc: 0.9195 | LR: 1.0e-03
[mean_pool s0] Epoch 09 | Train Loss: 0.1902, Train Acc: 0.9360 | Val Loss: 0.2008, Val 

In [8]:
for seed in SEEDS:
    run_experiment(
        tag="max_pool",
        seed=seed,
        device=device,
        pooling="max",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[max_pool s0] Epoch 01 | Train Loss: 2.0224, Train Acc: 0.3469 | Val Loss: 1.2158, Val Acc: 0.6437 | LR: 1.0e-03
[max_pool s0] Epoch 02 | Train Loss: 0.8123, Train Acc: 0.7478 | Val Loss: 0.4824, Val Acc: 0.8678 | LR: 1.0e-03
[max_pool s0] Epoch 03 | Train Loss: 0.4694, Train Acc: 0.8426 | Val Loss: 0.3979, Val Acc: 0.8333 | LR: 1.0e-03
[max_pool s0] Epoch 04 | Train Loss: 0.3250, Train Acc: 0.8881 | Val Loss: 0.3599, Val Acc: 0.8333 | LR: 1.0e-03
[max_pool s0] Epoch 05 | Train Loss: 0.3127, Train Acc: 0.8844 | Val Loss: 0.2458, Val Acc: 0.9195 | LR: 1.0e-03
[max_pool s0] Epoch 06 | Train Loss: 0.2411, Train Acc: 0.9200 | Val Loss: 0.2936, Val Acc: 0.8908 | LR: 1.0e-03
[max_pool s0] Epoch 07 | Train Loss: 0.2128, Train Acc: 0.9262 | Val Loss: 0.2730, Val Acc: 0.8908 | LR: 1.0e-03
[max_pool s0] Epoch 08 | Train Loss: 0.1934, Train Acc: 0.9446 | Val Loss: 0.2578, Val Acc: 0.9195 | LR: 1.0e-03
[max_pool s0] Epoch 09 | Train Loss: 0.1660, Train Acc: 0.9508 | Val Loss: 0.2987, Val Acc: 0.90

In [9]:
for seed in SEEDS:
    run_experiment(
        tag="last_pool",
        seed=seed,
        device=device,
        pooling="last",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

[last_pool s0] Epoch 01 | Train Loss: 1.6592, Train Acc: 0.4293 | Val Loss: 0.8178, Val Acc: 0.7644 | LR: 1.0e-03
[last_pool s0] Epoch 02 | Train Loss: 0.6029, Train Acc: 0.8130 | Val Loss: 0.4239, Val Acc: 0.8966 | LR: 1.0e-03
[last_pool s0] Epoch 03 | Train Loss: 0.4279, Train Acc: 0.8622 | Val Loss: 0.3547, Val Acc: 0.8621 | LR: 1.0e-03
[last_pool s0] Epoch 04 | Train Loss: 0.3064, Train Acc: 0.8942 | Val Loss: 0.2737, Val Acc: 0.9080 | LR: 1.0e-03
[last_pool s0] Epoch 05 | Train Loss: 0.2475, Train Acc: 0.9176 | Val Loss: 0.2456, Val Acc: 0.9253 | LR: 1.0e-03
[last_pool s0] Epoch 06 | Train Loss: 0.2382, Train Acc: 0.9225 | Val Loss: 0.2530, Val Acc: 0.9080 | LR: 1.0e-03
[last_pool s0] Epoch 07 | Train Loss: 0.2250, Train Acc: 0.9262 | Val Loss: 0.2080, Val Acc: 0.9368 | LR: 1.0e-03
[last_pool s0] Epoch 08 | Train Loss: 0.1682, Train Acc: 0.9508 | Val Loss: 0.2614, Val Acc: 0.9253 | LR: 1.0e-03
[last_pool s0] Epoch 09 | Train Loss: 0.1700, Train Acc: 0.9446 | Val Loss: 0.2645, Val 

## Results so far
Mean accuracy and macro-F1 over the five splits, with the half-width of the 95% confidence interval.

In [10]:
results = load_results(results_dir)
per_seed = per_seed_table(results)
summary = summary_table(per_seed)
summary[["tag", "perturbation", "n", "acc_mean", "acc_ci", "f1_mean", "f1_ci"]]

,tag,perturbation,n,acc_mean,acc_ci,f1_mean,f1_ci
0,full,,5,0.920000,0.029254,0.914877,0.031895
1,full_unpacked,,5,0.916571,0.011873,0.907525,0.013619
2,hp_00,,1,0.948571,NaN,0.946696,NaN
3,hp_01,,1,0.948571,NaN,0.945811,NaN
4,hp_02,,1,0.908571,NaN,0.907257,NaN
5,hp_03,,1,0.925714,NaN,0.920140,NaN
6,hp_04,,1,0.914286,NaN,0.908475,NaN
7,hp_05,,1,0.931429,NaN,0.926755,NaN
8,hp_06,,1,0.925714,NaN,0.921646,NaN
9,hp_07,,1,0.937143,NaN,0.933461,NaN
